In [53]:
import warnings 
warnings.filterwarnings(action='ignore')

In [54]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [55]:
df = pd.read_csv('melb_data.csv')
df.head(5)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [56]:
numerical_cols = df.select_dtypes(include=['int64', 'float64'])
categorical_cols = df.select_dtypes(include=['object'])

In [57]:
X, y = numerical_cols.drop(columns=['Price'], axis=1), numerical_cols['Price']

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [59]:
print('Total no. of Missing Values:')
print('-'*26)
print(X_train.isnull().sum())
print()
print('Avg Missing Values(%):')
print('-'*26)
print(X_train.isnull().mean()*100)

Total no. of Missing Values:
--------------------------
Rooms               0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                50
Landsize            0
BuildingArea     5129
YearBuilt        4286
Lattitude           0
Longtitude          0
Propertycount       0
dtype: int64

Avg Missing Values(%):
--------------------------
Rooms             0.000000
Distance          0.000000
Postcode          0.000000
Bedroom2          0.000000
Bathroom          0.000000
Car               0.460236
Landsize          0.000000
BuildingArea     47.210972
YearBuilt        39.451399
Lattitude         0.000000
Longtitude        0.000000
Propertycount     0.000000
dtype: float64


In [60]:
def score_dataset(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

cols_with_missing_values = [col for col in X_train.columns if X_train[col].isnull().any()]

reduced_X_train = X_train.drop(cols_with_missing_values, axis=1)
reduced_X_test = X_test.drop(cols_with_missing_values, axis=1)

print("Mean Absolute Error from dropping columns with Missing Values:")
print(score_dataset(reduced_X_train, reduced_X_test, y_train, y_test))

Mean Absolute Error from dropping columns with Missing Values:
177837.8946589946


In [61]:
imputer = SimpleImputer()
X_train_trf = imputer.fit_transform(X_train)
X_test_trf = imputer.transform(X_test)
print("Mean Absolute Error from Imputation:")
print(score_dataset(X_train_trf, X_test_trf, y_train, y_test))

Mean Absolute Error from Imputation:
171881.259907395


In [62]:
imputed_X_train = X_train.copy()
imputed_X_test = X_test.copy()

for col in cols_with_missing_values:
    imputed_X_train[col + '_missing'] = imputed_X_train[col].isnull()
    imputed_X_test[col + '_missing'] = imputed_X_test[col].isnull()


imputed_X_train_trf = imputer.fit_transform(imputed_X_train)
imputed_X_test_trf = imputer.transform(imputed_X_test)
print("Mean Absolute Error from Imputation while Track What Was Imputed:")
print(score_dataset(imputed_X_train_trf, imputed_X_test_trf, y_train, y_test))

Mean Absolute Error from Imputation while Track What Was Imputed:
171025.35443386633
